# Chapter 4.1-4.2: Text Features and Prompt Engineering

Goal: Understand why ML needs numeric features from text, practice traditional text extraction methods, then learn to write effective LLM prompts for structured data extraction.

### Topics:
- Why ML models can't consume raw text directly
- Keyword matching for sentiment analysis
- Regular expressions for extracting structured information
- Bag-of-words representation with CountVectorizer
- Writing effective LLM prompts (good vs bad)
- Requesting JSON output format from LLMs
- Few-shot prompting to improve extraction quality

In [ ]:
import pandas as pd
import numpy as np
import re
import json
from sklearn.feature_extraction.text import CountVectorizer

## Quick Recap

- **Feature engineering**: The process of creating numeric inputs for ML models from raw data
- **Text features**: Numeric representations of text data (word counts, sentiment scores, extracted fields)
- **Keyword matching**: Counting occurrences of known words to estimate properties like sentiment
- **Regular expressions (regex)**: Pattern-matching rules for extracting structured text (prices, dates, emails)
- **Bag-of-words**: Representing text as a vector of word counts, ignoring word order
- **Prompt engineering**: Crafting LLM instructions to get consistent, structured outputs
- **Few-shot prompting**: Including examples in the prompt to guide LLM behavior

## Data

We'll work with product reviews for electronics (coffee makers, headphones, chargers). All data is defined inline — no files to download.

**Note on simulated LLM output:** In a real workflow, you'd call an API (like OpenAI or Google Gemini) to get LLM responses. For this activity, we use pre-generated response strings so that everyone gets the same results and no API keys are needed.

In [ ]:
reviews = [
    {"id": 1, "product": "coffee maker", "text": "Absolutely love this coffee maker! Best purchase I've made all year. Brews perfect coffee every morning for $49.99."},
    {"id": 2, "product": "headphones", "text": "Terrible sound quality. Broke after two weeks. Complete waste of $79.99. Do NOT buy these."},
    {"id": 3, "product": "charger", "text": "It charges my phone. Nothing special but it works. Paid $12.99 which seems fair."},
    {"id": 4, "product": "coffee maker", "text": "The coffee tastes burnt and the machine is loud. Returned it after 3 days. Worst $89.99 I've ever spent."},
    {"id": 5, "product": "headphones", "text": "Great noise cancellation and comfortable fit. Battery lasts forever. Worth every penny of the $199.99."},
    {"id": 6, "product": "charger", "text": "Fast charging is nice but the cable feels cheap. Worried it won't last. $24.99 is okay I guess."},
    {"id": 7, "product": "coffee maker", "text": "Does the job but nothing more. Average coffee, average build quality. It's fine for $39.99."},
    {"id": 8, "product": "headphones", "text": "I wanted to love these SO much but the Bluetooth keeps disconnecting. Sound is amazing when it works though. $149.99 feels like a gamble."},
    {"id": 9, "product": "charger", "text": "Bought 3 of these at $9.99 each. Two work great, one was dead on arrival. Hit or miss quality."},
    {"id": 10, "product": "coffee maker", "text": "Upgraded from my old drip machine and WOW. The difference is night and day. $129.99 well spent."},
    {"id": 11, "product": "headphones", "text": "Sure, they work. If you like mediocre sound and uncomfortable ear cups. Save your money."},
    {"id": 12, "product": "charger", "text": "This charger literally saved my phone on a road trip. Charges super fast. Best $19.99 ever."},
    {"id": 13, "product": "coffee maker", "text": "Oh great, another coffee maker that leaks everywhere. Just what I needed. Thanks for nothing."},
    {"id": 14, "product": "headphones", "text": "Not bad for the price. Sound is decent, build quality is acceptable. Would buy again at $59.99."},
    {"id": 15, "product": "charger", "text": "Works exactly as described. Fast, reliable, good build. My third one from this brand. $14.99 is a steal."}
]

In [ ]:
# Simulated LLM responses — these are what an LLM *would* return for various prompts
SIMULATED_RESPONSES = {
    # Vague prompt responses (unstructured, inconsistent)
    (1, "vague"): "This review is positive. The customer likes the coffee maker.",
    (2, "vague"): "Negative sentiment. Bad product.",
    (8, "vague"): "Mixed feelings - the reviewer likes the sound but has connectivity issues. I'd say it's somewhat negative overall but with positive elements.",
    (13, "vague"): "The reviewer is being sarcastic and is clearly unhappy. Negative.",
    
    # Specific prompt responses (structured JSON)
    (1, "specific"): '{"sentiment": "positive", "confidence": 0.95, "key_phrases": ["absolutely love", "best purchase", "perfect coffee"]}',
    (2, "specific"): '{"sentiment": "negative", "confidence": 0.98, "key_phrases": ["terrible sound", "broke after two weeks", "do not buy"]}',
    (8, "specific"): '{"sentiment": "mixed", "confidence": 0.72, "key_phrases": ["wanted to love", "bluetooth disconnecting", "amazing when it works"]}',
    (13, "specific"): '{"sentiment": "negative", "confidence": 0.90, "key_phrases": ["leaks everywhere", "thanks for nothing"]}',

    # Zero-shot responses for tricky review (review 8 — mixed/sarcastic)
    (8, "zero_shot"): '{"sentiment": "positive", "confidence": 0.60, "reasoning": "The reviewer mentions amazing sound quality."}',
    
    # Few-shot responses for tricky review
    (8, "few_shot"): '{"sentiment": "mixed", "confidence": 0.78, "reasoning": "While the reviewer praises the sound quality, the Bluetooth disconnection issue and phrase wanting to love these suggests overall disappointment despite positive elements."}',

    # Zero-shot for sarcastic review (review 13)
    (13, "zero_shot"): '{"sentiment": "positive", "confidence": 0.55, "reasoning": "The reviewer says great and thanks."}',
    
    # Few-shot for sarcastic review
    (13, "few_shot"): '{"sentiment": "negative", "confidence": 0.92, "reasoning": "The phrases oh great and thanks for nothing are sarcastic. Combined with leaks everywhere, the review is clearly negative."}'
}

In [ ]:
# Job postings for prompt engineering exercise
job_postings = [
    "Senior Data Scientist needed at TechCorp. 5+ years experience required. Fully remote position. Must know Python, SQL, and machine learning.",
    "Junior Web Developer — StartupXYZ is hiring! 0-2 years experience. Hybrid work (3 days in office). React, JavaScript, CSS.",
    "Marketing Manager, BigBrand Inc. 7+ years in digital marketing. On-site only, NYC. SEO, content strategy, team leadership.",
    "ML Engineer (Mid-Level) at DataFlow. 3-5 years experience. Remote-first with optional office. PyTorch, MLOps, cloud platforms.",
    "Entry-Level Business Analyst at FinanceGroup. No experience required. On-site, Chicago. Excel, SQL, communication skills.",
    "Principal Software Engineer, MegaTech. 10+ years experience. Hybrid (2 days office). System design, distributed systems, Go/Rust.",
    "Data Analyst Intern at HealthData. Currently enrolled students. Remote summer internship. Python, Tableau, statistics.",
    "DevOps Lead — CloudNative Inc. 8+ years experience. Fully remote. Kubernetes, Terraform, AWS, CI/CD pipelines."
]

## Part 1: Traditional Approaches

Before LLMs existed, data scientists had to extract features from text using rule-based methods. These are still useful — they're fast, free, and predictable. Let's see what they can (and can't) do.

### 1. By hand — Manually label sentiment

Read the first 5 reviews and assign each a sentiment label: `"positive"`, `"negative"`, or `"mixed"`. This is your **ground truth** — the labels you'll compare automated methods against.

In [ ]:
# Print the first 5 reviews so you can read them
for r in reviews[:5]:
    print(f"Review {r['id']} ({r['product']}): {r['text']}")
    print()

In [ ]:
# Assign your manual labels here
manual_labels = {
    1: ...,  # "positive", "negative", or "mixed"
    2: ...,
    3: ...,
    4: ...,
    5: ...
}

### 2. By hand — Keyword-based sentiment function

Write a function `simple_sentiment(text)` that:
1. Counts how many **positive** words appear in the text (e.g., "love", "great", "best", "perfect", "amazing", "worth", "excellent")
2. Counts how many **negative** words appear (e.g., "terrible", "broke", "worst", "waste", "bad", "cheap", "mediocre")
3. Returns `"positive"` if positive > negative, `"negative"` if negative > positive, `"mixed"` if tied

Then compare your function's results on reviews 1-5 against your manual labels from exercise 1.

In [ ]:
def simple_sentiment(text):
    """Classify sentiment based on keyword counts."""
    text_lower = text.lower()
    
    positive_words = ["love", "great", "best", "perfect", "amazing", "worth", "excellent", "wow"]
    negative_words = ["terrible", "broke", "worst", "waste", "bad", "cheap", "mediocre", "loud"]
    
    # Count positive and negative word occurrences
    pos_count = ...
    neg_count = ...
    
    # Return sentiment based on counts
    ...

In [ ]:
# Compare keyword-based results to your manual labels
print("Review | Manual Label | Keyword Label | Match?")
print("-" * 50)
for r in reviews[:5]:
    keyword_label = simple_sentiment(r["text"])
    manual = manual_labels[r["id"]]
    match = "Yes" if keyword_label == manual else "No"
    print(f"  {r['id']}    | {manual:10s} | {keyword_label:13s} | {match}")

**Your observation:** Which reviews did the keyword approach get wrong? Why did it fail?

(Write your answer here)

### 3. By hand — Extract prices with regex

Use `re.findall()` to extract all prices (patterns like `$XX.XX`) from each review. Store the results in a dictionary mapping review ID to a list of prices found.

In [ ]:
# Write a regex pattern to match prices like $49.99, $9.99, $199.99
price_pattern = ...

# Extract prices from each review
prices_found = {}
for r in reviews:
    prices = re.findall(price_pattern, r["text"])
    prices_found[r["id"]] = prices

# Display results
for review_id, prices in prices_found.items():
    if prices:
        print(f"Review {review_id}: {prices}")

### 4. Use AI — Bag-of-words with CountVectorizer

Use your AI assistant to implement bag-of-words using scikit-learn's `CountVectorizer`. Fit it on all 15 review texts and display the resulting feature matrix as a DataFrame. Use `max_features=20` to keep it manageable.

After generating the code, look at the DataFrame and answer: what information is lost in this representation?

In [ ]:
# Use AI to help implement bag-of-words
# Extract just the text from each review
review_texts = [r["text"] for r in reviews]

# Fit CountVectorizer with max_features=20
...

# Convert to DataFrame with feature names as columns
...

**Your observation:** What information about the reviews is lost in the bag-of-words representation?

(Write your answer here)

## Part 2: Prompt Engineering

Traditional methods are fast and free, but they miss nuance. LLMs can understand context, sarcasm, and complex meaning — but only if you prompt them well. Let's practice writing effective prompts.

### 5. By hand — Compare good vs bad prompts

Below are two prompts for sentiment analysis. Read them both, then:
1. Predict what kind of response each would produce
2. Explain which is better and why

In [ ]:
vague_prompt = """What do you think about this review?

Review: {review_text}"""

specific_prompt = """Analyze the sentiment of the following product review.

Return your response as JSON with exactly these fields:
- "sentiment": one of "positive", "negative", or "mixed"
- "confidence": a float between 0 and 1
- "key_phrases": a list of 2-4 phrases from the review that support your classification

Review: {review_text}

Respond with only the JSON, no additional text."""

print("=== VAGUE PROMPT ===")
print(vague_prompt.format(review_text=reviews[0]["text"]))
print()
print("=== SPECIFIC PROMPT ===")
print(specific_prompt.format(review_text=reviews[0]["text"]))

In [ ]:
# Let's see what each prompt actually produces (simulated responses)
print("=== Vague prompt response (Review 1) ===")
print(SIMULATED_RESPONSES[(1, "vague")])
print()
print("=== Specific prompt response (Review 1) ===")
print(SIMULATED_RESPONSES[(1, "specific")])

**Your analysis:** Why is the specific prompt better for a data pipeline? Consider: consistency, parseability, and what happens when you process 10,000 reviews.

(Write your answer here)

### 6. By hand — Parse a messy response

The vague prompt gives us natural language responses that vary in format. Try to extract just the sentiment label from each vague response below. How reliable is this?

In [ ]:
vague_responses = [
    SIMULATED_RESPONSES[(1, "vague")],
    SIMULATED_RESPONSES[(2, "vague")],
    SIMULATED_RESPONSES[(8, "vague")],
    SIMULATED_RESPONSES[(13, "vague")],
]

# Try to extract sentiment from each vague response
# Hint: you might check if certain words appear in the response
for i, response in enumerate(vague_responses):
    print(f"Response: {response}")
    
    # Write code to extract sentiment label from this unstructured text
    extracted_sentiment = ...
    
    print(f"Extracted: {extracted_sentiment}")
    print()

**Your observation:** What makes parsing these vague responses difficult? What could go wrong at scale?

(Write your answer here)

### 7. By hand — Write an extraction prompt for job postings

Write a prompt that instructs an LLM to extract structured information from a job posting. The output should be JSON with these 4 fields:
- `title`: the job title
- `category`: one of "engineering", "data_science", "marketing", "business", "other"
- `experience_level`: one of "entry", "mid", "senior", "principal"
- `remote_policy`: one of "remote", "hybrid", "on_site"

In [ ]:
# Write your extraction prompt
job_extraction_prompt = """
...
"""

# Test it by printing what it looks like with a real job posting
print(job_extraction_prompt.format(job_posting=job_postings[0]))

**Your checklist:** Does your prompt include:
- [ ] Clear task description
- [ ] Exact field names
- [ ] Valid values for each field
- [ ] Instruction to return only JSON
- [ ] A placeholder for the input text

### 8. Use AI — Build a few-shot prompt function

Use your AI assistant to write a function `build_few_shot_prompt(text, examples, task)` that:
1. Takes a review text, a list of example (text, label) pairs, and a task description
2. Builds a prompt that includes the examples before the actual review
3. Returns the complete prompt string

Use 3 examples: one positive, one negative, one mixed.

In [ ]:
# Define 3 examples for few-shot prompting
sentiment_examples = [
    ("This product is fantastic! Works perfectly every time.", 
     '{"sentiment": "positive", "confidence": 0.95, "reasoning": "Strong positive language with no complaints."}'),
    ("Broke on day one. Terrible quality. Want my money back.", 
     '{"sentiment": "negative", "confidence": 0.97, "reasoning": "Product failure, strong negative language, refund request."}'),
    ("Good features but the battery dies too fast. Hard to recommend.", 
     '{"sentiment": "mixed", "confidence": 0.80, "reasoning": "Acknowledges positives but significant negative outweighs them."}'),
]

# Use AI to write this function
def build_few_shot_prompt(text, examples, task):
    """Build a few-shot prompt with examples.
    
    Args:
        text: The review to analyze
        examples: List of (input_text, expected_output) tuples
        task: Description of the task
    
    Returns:
        Complete prompt string with examples
    """
    ...

# Test it
test_prompt = build_few_shot_prompt(
    reviews[7]["text"],  # Review 8 — the tricky mixed one
    sentiment_examples,
    "Analyze the sentiment of the following product review. Return JSON with sentiment, confidence, and reasoning."
)
print(test_prompt)

### 9. By hand — Compare zero-shot vs few-shot on tricky reviews

Let's compare how zero-shot and few-shot prompts handle two tricky reviews:
- **Review 8** (mixed): "I wanted to love these SO much but the Bluetooth keeps disconnecting..."
- **Review 13** (sarcastic): "Oh great, another coffee maker that leaks everywhere..."

In [ ]:
# Review 8: Mixed sentiment
print("=== Review 8 (Mixed) ===")
print(f"Text: {reviews[7]['text']}")
print()
print("Zero-shot response:")
print(SIMULATED_RESPONSES[(8, "zero_shot")])
print()
print("Few-shot response:")
print(SIMULATED_RESPONSES[(8, "few_shot")])
print()
print()
# Review 13: Sarcasm
print("=== Review 13 (Sarcastic) ===")
print(f"Text: {reviews[12]['text']}")
print()
print("Zero-shot response:")
print(SIMULATED_RESPONSES[(13, "zero_shot")])
print()
print("Few-shot response:")
print(SIMULATED_RESPONSES[(13, "few_shot")])

**Your analysis:**

1. For review 8, which response is more accurate? What did few-shot get right that zero-shot missed?

(Write your answer here)

2. For review 13, the zero-shot response misidentified sarcasm. How did the few-shot examples help?

(Write your answer here)

3. What kind of examples would you include in a few-shot prompt to handle sarcasm better?

(Write your answer here)

## Discussion

What text information can't be captured by keywords or bag-of-words that an LLM could extract? What are the tradeoffs — cost, consistency, speed, interpretability?

(Discuss with a neighbor)